In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings
import torch

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R


In [2]:
sample_name="151671"
n_clusters = 5 if sample_name in [str(151669 + x) for x in range(4)] else 7
data_dir  = '/home/pxy/home/pxy/data/DLPFC/st/'

# data_file = f"../dataset/DLPFC/{sample_name}.h5ad"
# adata = sc.read_h5ad(data_file)
adata = sc.read_visium(data_dir + sample_name)
adata.var_names_make_unique()
print(adata)

AnnData object with n_obs × n_vars = 4110 × 33538
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'


In [3]:
import pandas as pd

meta = pd.read_csv(
    data_dir+sample_name+"/metadata.tsv",
    sep="\t"
)

meta = meta.set_index("barcode")
adata.obs["Region"] = meta.loc[
    adata.obs_names,
    "layer_guess_reordered"
]
print(adata.obs["Region"].value_counts())
print("NA spots:", adata.obs["Region"].isna().sum())

Region
Layer3    1918
Layer6     760
Layer5     721
WM         449
Layer4     245
Name: count, dtype: int64
NA spots: 17


In [4]:
adata = spCLUE.preprocess(adata)
adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
g_spatia = spCLUE.prepare_graph(adata, "spatial")
g_expr = spCLUE.prepare_graph(adata, "expr", metric="euclidean")
graph_dict = {"spatial": g_spatia, "expr":g_expr}

normalized data ---------------->


正在构建图: spatial, 使用度量: cosine ...
  -> 使用空间坐标 (euclidean)
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
spatial graph created successfully <----

正在构建图: expr, 使用度量: euclidean ...
  -> 使用 PCA 表达特征
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
expr graph created successfully <----



In [5]:
model = spCLUE.spCLUE_TwoStage(
    adata.obsm["X_pca"], 
    graph_dict, 
    n_clusters=n_clusters,
    pretrain_epochs=100,   # 预训练200轮
    finetune_epochs=100,   # 训练300轮
    gamma=1.0,             # 重构损失权重
    beta=0.0,              # 聚类损失权重=0 (关键!)
    kappa=1.0,             # 对比损失权重
    freeze_encoder=True,   # 冻结预训练编码器
)

In [6]:
pred, embed = model.train()

Stage 1: Pre-training Shared Encoder


Pretrain:   0%|          | 0/100 [00:00<?, ?it/s]

Pretrain:  26%|██▌       | 26/100 [00:01<00:03, 22.35it/s]

  Pretrain Epoch 10: Rec Loss = 10.722700
  Pretrain Epoch 20: Rec Loss = 10.611124
  Pretrain Epoch 30: Rec Loss = 10.461108


Pretrain:  50%|█████     | 50/100 [00:02<00:01, 45.74it/s]

...

  Train Epoch 80: Rec Loss = 18.067001, Contrast Loss = 7.384624, Cluster Loss = 0.000000


Finetune:  91%|█████████ | 91/100 [00:09<00:00, 10.54it/s]

  Train Epoch 90: Rec Loss = 18.012844, Contrast Loss = 7.333366, Cluster Loss = 0.000000


Finetune: 100%|██████████| 100/100 [00:10<00:00,  9.83it/s]

  Train Epoch 100: Rec Loss = 17.984184, Contrast Loss = 7.307457, Cluster Loss = 0.000000

  Finetune Epoch 100:
    Total Loss   = 17.9842
    Rec Loss     = 10.6767
    Contrast Loss= 7.3075
    Gate: spatial=0.558±0.128, expr=0.442±0.128

✓ Finetune finished (max epochs reached)
  [Gate Stats] spatial=0.558±0.128, expr=0.442±0.128


In [7]:
# ========== 聚类 ==========
adata.obsm["spCLUE_twostage"] = embed
spCLUE.clustering(adata, n_clusters, key="spCLUE_twostage", refinement=True)

# ========== 评估 ==========
adata_filtered = adata[adata.obs.Region.notna()]
ARI = adjusted_rand_score(adata_filtered.obs["Region"], 
                          adata_filtered.obs["mclust_refined"])
print(f"\nFinal ARI on {sample_name}: {ARI:.4f}")

R[write to console]:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.1.2
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%

Final ARI on 151671: 0.6053


In [ ]:
# adata.obs["spCLUE"] = adata.obs["mclust_refined"]
# sc.pl.spatial(adata, color=["Region", "spCLUE"], title=["Manual Annotation", f"spCLUE (ARI={round(ARI, 2)})"])

: 